In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class InteractionLayer(nn.Module):
    def __init__(
        self,
        n_features,
        activation=torch.sigmoid,
        adjacency_activation=torch.sigmoid
    ):
        super().__init__()

        self.n_features = n_features

        # Outer activation
        self.activation = activation

        # Activation used for A = sigma(W)
        self.adjacency_activation = adjacency_activation

        # Learnable adjacency logits
        self.W = nn.Parameter(
            torch.randn(n_features, n_features) * 0.1
        )

        # Learnable feature parameters
        self.k = nn.Parameter(torch.ones(n_features))
        self.tau = nn.Parameter(torch.zeros(n_features))

    @property
    def A(self):
        """
        Soft adjacency matrix:
            A = sigma(W)
        """
        return self.adjacency_activation(self.W)

    def forward(self, s):
        """
        s shape:
            (batch_size, n_features)

        returns:
            s_tilde with same shape
        """

        # Compute adjacency
        A = self.A

        # k_j * (s_j - tau_j)
        weighted = self.k * (s - self.tau)

        # Sum_j A_ji * ...
        interaction = weighted @ A

        # sigma(...)
        gate = self.activation(interaction)

        # s_i * sigma(...)
        s_tilde = s * gate

        return s_tilde
    
class FullModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.layer = InteractionLayer(n_features)

        self.readout = nn.Linear(n_features, 1)

    def forward(self, s, A):

        s_tilde = self.layer(s, A)

        score = self.readout(s_tilde)

        return score

In [ ]:
n_features = 12
batch_size = 64

model = FullModel(n_features)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Dummy data
x = torch.randn(batch_size, n_features)
y = torch.randn(batch_size, 1)

# Sparsity coefficient
lambda_sparse = 1e-2

for step in range(1000):

    optimizer.zero_grad()

    pred = model(x)

    # Main prediction loss
    prediction_loss = F.mse_loss(pred, y)

    # Sparsity penalty on adjacency matrix
    sparsity_loss = model.layer.A.abs().mean()

    # Total objective
    loss = prediction_loss + lambda_sparse * sparsity_loss

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        print(
            f"step={step}",
            f"prediction={prediction_loss.item():.4f}",
            f"sparsity={sparsity_loss.item():.4f}",
            f"total={loss.item():.4f}"
        )